# Attention Head Interpretability

Per-head interpretability tools: function classification heuristics,
entropy-behavior mapping, QK/OV readable summaries, importance vs
interpretability tradeoff, and head summary cards.

## Why This Matters

Attention head interpretability reveals how heads route information between positions. Understanding attention mechanics is central to reverse-engineering transformer circuits, as attention patterns determine what information each component can access and transform.

**Key references:**
- [Elhage et al. (2021) "A Mathematical Framework for Transformer Circuits"](https://transformer-circuits.pub/2021/framework/index.html) — Foundational framework for attention head and residual stream analysis
- [Olsson et al. (2022) "In-context Learning and Induction Heads"](https://arxiv.org/abs/2209.11895) — Discovers induction heads and training phase transitions

In [ ]:
import jax
import jax.numpy as jnp
import numpy as np
from irtk import HookedTransformer, HookedTransformerConfig
from irtk.attention_head_interpretability import (
    head_function_classification,
    entropy_behavior_mapping,
    qk_ov_summary,
    importance_interpretability_tradeoff,
    head_summary_card,
)

In [ ]:
cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)
tokens = jnp.array([0, 5, 10, 15, 20, 25, 30])
metric_fn = lambda logits: float(logits[-1, 0] - logits[-1, 1])
print("Model ready")

## Head Function Classification

Categorize each head using behavioral heuristics: previous-token,
induction, self-attention, BOS attention, etc.

In [ ]:
result = head_function_classification(model, tokens)
print("Function counts:")
for func, count in sorted(result['function_counts'].items(), key=lambda x: -x[1]):
    print(f"  {func}: {count}")
print(f"\nHead classifications:")
for l in range(cfg.n_layers):
    for h in range(cfg.n_heads):
        label = result['classifications'][(l, h)]
        conf = result['confidence'][(l, h)]
        print(f"  L{l}H{h}: {label} (confidence={conf:.3f})")

## Entropy-Behavior Mapping

In [ ]:
result = entropy_behavior_mapping(model, tokens)
for l in range(cfg.n_layers):
    for h in range(cfg.n_heads):
        cat = result['entropy_categories'][(l, h)]
        ent = result['head_entropy'][l, h]
        focus = result['focus_positions'][(l, h)]
        print(f"L{l}H{h}: {cat} (entropy={ent:.3f}), focus={focus}")

## QK/OV Circuit Summaries

In [ ]:
for l in range(min(2, cfg.n_layers)):
    for h in range(min(2, cfg.n_heads)):
        result = qk_ov_summary(model, layer=l, head=h)
        print(f"L{l}H{h}: QK rank={result['qk_rank']:.1f}, OV rank={result['ov_rank']:.1f}")
        print(f"  QK SVs: {[f'{s:.3f}' for s in result['qk_singular_values'][:3]]}")
        print(f"  OV SVs: {[f'{s:.3f}' for s in result['ov_singular_values'][:3]]}")

## Importance vs Interpretability Tradeoff

In [ ]:
result = importance_interpretability_tradeoff(model, tokens, metric_fn)
print(f"Correlation: {result['tradeoff_correlation']:.4f}")
print(f"\nImportant but hard to interpret:")
for l, h in result['important_uninterpretable'][:5]:
    print(f"  L{l}H{h}: imp={result['importance'][l,h]:.4f}, interp={result['interpretability'][l,h]:.4f}")
print(f"\nInterpretable but unimportant:")
for l, h in result['unimportant_interpretable'][:5]:
    print(f"  L{l}H{h}: imp={result['importance'][l,h]:.4f}, interp={result['interpretability'][l,h]:.4f}")

## Head Summary Cards

In [ ]:
card = head_summary_card(model, tokens, layer=0, head=0, metric_fn=metric_fn)
print(f"=== Head L{card['identity'][0]}H{card['identity'][1]} ===")
print(f"Function: {card['function_type']}")
print(f"Entropy: {card['entropy_category']} ({card['mean_entropy']:.3f})")
print(f"Clarity: {card['clarity']:.3f}")
print(f"Importance: {card['importance']:.4f}")
print(f"QK rank: {card['qk_rank']:.1f}, OV rank: {card['ov_rank']:.1f}")
print(f"Focus positions: {card['top_attended_positions']}")